# 03 · Build the evaluation set (English → Hindi, Portuguese)

**Runtime: T4 GPU.** Translation runs on the GPU; everything else would run on CPU.

Turns public English datasets into **parallel items**: the same question in English, Hindi and Portuguese. Every later step (generation, agents, analysis) reads these items.

| Step | Input | Output on Drive (`MyDrive/mpac_hi_pt/`) |
|---|---|---|
| 1 Fetch | PLURAL, TruthfulQA, UltraFeedback (Hugging Face) | `build/source_items.jsonl`: English items |
| 2 Translate | English stems, options, instruction | `translations/madlad400-3b-mt/{hi,pt}.jsonl` |
| 3 Assemble | 1 + 2 | `items/items.jsonl`: one prompt per language per item |
| 4 Verify | 3 | `verification/{hi,pt}.csv`: sheets for a fluent speaker |
| 5 Split | 3 | `splits/splits.json`: tune / test per item |

Every step is cached on Drive. If Colab disconnects, re-run the notebook and it continues where it stopped.

In [ ]:
# Clone (or update) the repo and install it. Re-run after a runtime reset.
import os, subprocess
REPO_DIR = "/content/mpac"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/hurshprasad/Multi-Agent-Diagnosis-of-Multilingual-Pluralistic-Alignment.git $REPO_DIR
else:
    !git -C $REPO_DIR pull --ff-only
%cd $REPO_DIR
!pip install -q -e ".[gpu]"
import sys
sys.path.insert(0, f"{REPO_DIR}/src")  # editable installs aren't always visible to a running kernel

In [ ]:
QUICK = True   # first run: 30 items, small model, minutes. Set False for the full ~500-item run.
CONFIGS = ["configs/hi_pt.yaml"] + (["configs/hi_pt_quick.yaml"] if QUICK else [])

from google.colab import drive
drive.mount("/content/drive")
from mpac import load_config
from mpac.paths import resolve_root, subdir
cfg = load_config(*CONFIGS)
print("languages:", cfg["languages"], "| artifacts ->", resolve_root(cfg))

## Steps 1–3 · Fetch, translate, assemble

- **Fetch:** samples items from each source, in the counts set under `build:` in the config.
- **Translate:** the question text, each answer option and the instruction ("Answer with only the letter…") are translated separately with MADLAD-400. The option letters A, B, C… never go through translation, so they are identical in every language.
- **Assemble:** rebuilds the full prompt in each language.

Quick run: about 5 minutes, mostly downloading the MT model. Full run: about 30–60 minutes on a T4.

In [ ]:
from collections import Counter
from mpac.data.build import build_dataset

items = build_dataset(cfg)
print(len(items), "items")
print("by source:", dict(Counter(it.meta["source"] for it in items)))
print("languages per item:", dict(Counter(len(it.prompts) for it in items)))

In [ ]:
# Look at one item per source in every language: this is exactly what the model will be asked.
for src in ["wvs", "truthfulqa", "plural", "ultrafeedback"]:
    it = next((i for i in items if i.meta["source"] == src), None)
    if it is None:
        continue
    print(f"=============== {it.item_id}  ({it.objective})")
    for lang, prompt in it.prompts.items():
        print(f"--- {lang}\n{prompt[:600]}\n")

## Step 4 · Verify translations

This writes `verification/hi.csv` and `verification/pt.csv`, covering the **same** items in both languages, spread over the three objectives. Open each one in Google Sheets (Drive → right-click → Open with → Google Sheets). A fluent speaker then fills in, for each row:
- `ok`: `yes` if the translation asks the same thing as the English, `no` if it doesn't.
- `corrected`: optional. The full corrected prompt, if the translation is wrong.

To save: **File → Download → CSV**, then upload it back to the same folder with the same name, replacing the old file.

**Do this before generating (notebook 01).** A corrected prompt only takes effect for items that haven't been generated yet. For Portuguese, also check it reads as Brazilian Portuguese (D9).

In [ ]:
from mpac.data.verification import export_verification, verification_ids

ids = verification_ids(items, cfg["data"]["n_verified_per_lang"], cfg["project"]["seed"])
for lang in cfg["languages"]["targets"]:
    path = subdir(cfg, "verification") / f"{lang}.csv"
    try:
        print(lang, "->", export_verification(items, lang, ids, path), "rows at", path)
    except FileExistsError as e:
        print(lang, "->", e)   # already has answers: kept as is

In [ ]:
# After the sheets are filled: how many translations were accepted, corrected or rejected.
from mpac.data.loaders import load_items
from mpac.data.verification import apply_verification

for lang in cfg["languages"]["targets"]:
    path = subdir(cfg, "verification") / f"{lang}.csv"
    print(lang, apply_verification(load_items(cfg), lang, path))

## Step 5 · Tune / test split

Each item goes into `tune` (about 1/3) or `test`, once, and the assignment is saved. Human labels on `tune` items may be used to set thresholds; reported results only use `test`. Every language version of an item lands in the same split.

In [ ]:
from mpac.data.splits import get_or_create_splits

splits = get_or_create_splits(cfg, items)
print(Counter(splits.values()))
print(Counter((it.objective, splits[it.item_id]) for it in items))

## Next

Open `01_generate.ipynb` and set the **same** `QUICK` value. It reads `items/items.jsonl` and writes model responses to `generations/`.